# Experimentos

## Importando requisitos

In [1]:
import pandas
import os
import numpy
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

## Abrindo a planilha

In [2]:
planilha = pandas.read_csv('../Dataset/DataSummary.csv')
planilha

,ID,Type,Name,Train,Test,Class,Length,ED (w=0),DTW (learned_w),DTW (w=100),Default rate,Data donor/editor
0,1,Image,Adiac,390,391,37,176,0.3887,0.3913 (3),0.3964,0.9591,A. Jalba
1,2,Image,ArrowHead,36,175,3,251,0.2000,0.2000 (0),0.2971,0.6057,L. Ye & E. Keogh
2,3,Spectro,Beef,30,30,5,470,0.3333,0.3333 (0),0.3667,0.8000,K. Kemsley & A. Bagnall
3,4,Image,BeetleFly,20,20,2,512,0.2500,0.3000 (7),0.3000,0.5000,J. Hills & A. Bagnall
4,5,Image,BirdChicken,20,20,2,512,0.4500,0.3000 (6),0.2500,0.5000,J. Hills & A. Bagnall
...,...,...,...,...,...,...,...,...,...,...,...,...
123,124,Spectrum,SemgHandMovementCh2,450,450,6,1500,0.6311,0.3622 (1),0.4156,0.8333,C.-C. M. Yeh
124,125,Spectrum,SemgHandSubjectCh2,450,450,5,1500,0.5956,0.2000 (3),0.2733,0.8000,C.-C. M. Yeh
125,126,Sensor,ShakeGestureWiimoteZ,50,50,10,Vary,0.4000,0.1600 (6),0.1400,0.9000,J. Guna
126,127,Simulated,SmoothSubspace,150,150,3,15,0.0933,0.0533 (1),0.1733,0.6667,X. Huang


## Filtrando na planilha os datasets selecionados

In [3]:
planilha = planilha[planilha['ID'].between(97, 128)]
planilha

,ID,Type,Name,Train,Test,Class,Length,ED (w=0),DTW (learned_w),DTW (w=100),Default rate,Data donor/editor
96,97,EOG,EOGVerticalSignal,362,362,12,1250,0.5580,0.5249 (2),0.5525,0.9144,E. Keogh & H. A. Dau
97,98,Spectro,EthanolLevel,504,500,4,1751,0.7260,0.7180 (1),0.7240,0.7480,A. Bagnall
98,99,Sensor,FreezerRegularTrain,150,2850,2,301,0.1951,0.0930 (1),0.1011,0.5000,REFIT project
99,100,Sensor,FreezerSmallTrain,28,2850,2,301,0.3242,0.3242 (0),0.2411,0.5000,REFIT project
100,101,HRM,Fungi,18,186,18,201,0.1774,0.1774 (0),0.1613,0.8978,W. Fonzi
101,102,Trajectory,GestureMidAirD1,208,130,26,Vary,0.4231,0.3615 (5),0.4308,0.9615,H. A. Dau
102,103,Trajectory,GestureMidAirD2,208,130,26,Vary,0.5077,0.4000 (6),0.3923,0.9615,H. A. Dau
103,104,Trajectory,GestureMidAirD3,208,130,26,Vary,0.6538,0.6231 (1),0.6769,0.9615,H. A. Dau
104,105,Sensor,GesturePebbleZ1,132,172,6,Vary,0.2674,0.1744 (2),0.2093,0.8140,I. Maglogiannis
105,106,Sensor,GesturePebbleZ2,146,158,6,Vary,0.3291,0.2215 (6),0.3291,0.8101,I. Maglogiannis


In [4]:
def formatar_dataset(df: pandas.DataFrame) -> pandas.DataFrame:
    """
    Realiza a formatação do dataset.
    :param df: Dataset a ser formatado.
    :return: Dataset formatado.
    """
    classe = df.iloc[:, 0]
    series = df.iloc[:, 1:]

    df_novo = pandas.DataFrame({
        'classe': classe,
        'SérieTemporal': list(series.to_numpy())
    })

    return df_novo

In [5]:
from app.model.DynamicTimeWarping import DynamicTimeWarping
from app.model.DerivativeDynamicTimeWarping import DerivativeDynamicTimeWarping
from app.model.LongestCommonSubsequence import LongestCommonSubsequence
from app.model.SoftDynamicTimeWarping import SoftDynamicTimeWarping

def aplicar_algoritmos_series_temporais(dataset: pandas.DataFrame, indice_serie_referencia: int = 0) -> pandas.DataFrame:
    """
    Aplica algoritmos às séries temporais.
    :param dataset: Dataset.
    :param indice_serie_referencia: Índice da série temporal de referência.
    :return: Dataset com as distâncias das séries temporais.
    """
    # Instanciando algoritmos
    dtw = DynamicTimeWarping()
    ddtw = DerivativeDynamicTimeWarping()
    lcs = LongestCommonSubsequence()
    soft_dtw = SoftDynamicTimeWarping()

    # Definindo série de referência
    serie_referencia = dataset.iloc[indice_serie_referencia]["SérieTemporal"]

    # Executando algoritmos e adicionando ao Dataset
    print('Executando Dynamic Time Warping')
    dataset['dtw'] = dataset['SérieTemporal'].apply(
        lambda s: dtw.obter_distancia(s, serie_referencia)
    )
    print('Finalizado o Dynamic Time Warping')

    print('Executando o Derivative Dynamic Time Warping')
    dataset['ddtw'] = dataset['SérieTemporal'].apply(
        lambda s: ddtw.obter_distancia(s, serie_referencia)
    )
    print('Finalizado o Derivative Dynamic Time Warping')

    print('Executando o Longest Common Subsequence')
    dataset['lcs'] = dataset['SérieTemporal'].apply(
        lambda s: lcs.obter_distancia(s, serie_referencia)
    )
    print('Finalizado o Longest Common Subsequence')

    print('Executando o Soft Dynamic Time Warping')
    dataset['soft-dtw'] = dataset['SérieTemporal'].apply(
        lambda s: soft_dtw.obter_distancia(s, serie_referencia)
    )
    print('Finalizado o Soft Dynamic Time Warping')

    return dataset

In [6]:
import os
import pathlib
import json

def salvar_como_json(dados: object, filepath: str):
    """
    Salva um objeto como JSON.
    :param dados: Objeto a ser salvo.
    :param filepath: Caminho do arquivo onde será salvo o objeto.
    :return:
    """

    # Criando o diretório pai, caso ele não exista
    os.makedirs(pathlib.Path(filepath).parent, exist_ok=True)

    with open(filepath, 'w', encoding='utf-8') as arquivo:
        json.dump(dados, arquivo, indent=4, ensure_ascii=False)

In [7]:
def carregar_arquivo_json(filepath: str) -> object:
    """
    Carrega um arquivo JSON.
    :param filepath: Caminho do arquivo onde será lido o objeto.
    :return: Objeto carregado a partir do arquivo JSON.
    """
    with open(filepath, 'r', encoding='utf-8') as arquivo:
        return json.load(arquivo)

In [8]:
def separar_x_y(dataset: pandas.DataFrame) -> tuple[list, list]:
    """
    Separa o conjunto de treino do conjunto de teste
    :param dataset: Dataset de entrada.
    :return: Tupla com os cunjuntos de treino e teste.
    """
    x = (dataset['SérieTemporal'] + dataset['dtw'] + dataset['ddtw'] + dataset['lcs'] + dataset['soft-dtw']).tolist()

    y = dataset['classe'].tolist()

    return x, y

In [9]:
import sklearn.svm

## Realizando experimentos no dataset da tabela toda

Definindo o tamanho máximo da recursão permitida

In [10]:
from sklearn.metrics import accuracy_score
from sklearn.base import ClassifierMixin


def treinar_modelo(planilha: pandas.DataFrame, modelos: list[ClassifierMixin]) -> list[ClassifierMixin]:
    lista_nomes_datasets = planilha.iloc[:]['Name']
    for nome_dataset in lista_nomes_datasets:
        # Obtendo os caminhos do dataset de treino e teste
        print('Obtendo os caminhos do dataset de treino e teste')
        diretorio_dataset = os.path.join('../Dataset/UCRArchive_2018', nome_dataset)
        caminho_arquivo_treino = os.path.join(diretorio_dataset, '{}_TRAIN.tsv'.format(nome_dataset))
        caminho_arquivo_teste = os.path.join(diretorio_dataset, '{}_TEST.tsv'.format(nome_dataset))

        # Abrindo datasets
        print('Abrindo datasets')
        dataset_treino = pandas.read_csv(caminho_arquivo_treino, sep='\t', header=None)
        dataset_teste = pandas.read_csv(caminho_arquivo_teste, sep='\t', header=None)

        # Formatando datasets
        print('Formatando datasets')
        dataset_treino = formatar_dataset(dataset_treino)
        dataset_teste = formatar_dataset(dataset_teste)

        # Aplicando algoritmos nos datasets
        print('Aplicando algoritmos nos datasets')
        dataset_treino = aplicar_algoritmos_series_temporais(dataset_treino)
        dataset_teste = aplicar_algoritmos_series_temporais(dataset_teste)

        # Criando conjunto de treino e teste
        print('Criando conjunto de treino e teste')
        x_train, y_train = separar_x_y(dataset_treino)
        x_test, y_test = separar_x_y(dataset_teste)

        # Treinando cada modelo
        for modelo in modelos:

            # Treinando modelo
            print('Treinando modelo')
            modelo.fit(x_train, y_train)

            # Calculando acurácia
            print('Calculando acurácia')
            y_pred = modelo.predict(x_test)
            acuracia = accuracy_score(y_test, y_pred)
            print('Acurácia: {}'.format(acuracia))

    return modelos

In [ ]:
from sklearn.neighbors._classification import KNeighborsClassifier
from sklearn.svm import SVC

knn = KNeighborsClassifier()
svm = SVC()
treinar_modelo(planilha, [knn, svm])

Obtendo os caminhos do dataset de treino e teste
Abrindo datasets
Formatando datasets
Aplicando algoritmos nos datasets
Executando Dynamic Time Warping


Salvando modelo

In [ ]:
import pickle

def salvar_arquivo_pickle(dados: object, filepath: str):
    """
    Salva um objeto como Pickle.
    :param dados: Objeto a ser salvo.
    :param filepath: Caminho do arquivo onde será salvo o objeto.
    :return:
    """
    # Criando o diretório pai, caso ele não exista
    os.makedirs(pathlib.Path(filepath).parent, exist_ok=True)

    with open(filepath, 'wb') as arquivo:
        pickle.dump(dados, arquivo)

In [ ]:
def carregar_arquivo_pickle(filepath: str) -> object:
    """
    Carrega um arquivo Pickle.
    :param filepath: Caminho do arquivo onde seré lido o objeto.
    :return: Objeto carregado a partir do arquivo Pickle.
    """
    with open(filepath, 'rb') as arquivo:
        return pickle.load(arquivo)

In [ ]:
salvar_arquivo_pickle(dados=svm, filepath='../Models/svm.pkl')